<a href="https://colab.research.google.com/github/Nanzz1112/Tugas2_AI/blob/main/prediksi_emas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Prediksi Arah Pergerakan Harga Emas
# Algoritma: Random Forest dan Logistic Regression

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# 1. Membaca dataset
# Letakkan file CSV di folder yang sama dengan program ini.
nama_file = "gold.csv"

try:
    data = pd.read_csv(nama_file)
except FileNotFoundError:
    print(f"File '{nama_file}' tidak ditemukan.")
    print("Unduh dataset Kaggle, lalu ubah nama file menjadi gold.csv.")
    raise SystemExit

# Merapikan nama kolom agar lebih mudah dicocokkan
data.columns = [str(kolom).strip().lower() for kolom in data.columns]

print("\n=== DATASET HARGA EMAS ===")
print("Jumlah baris dan kolom:", data.shape)
print("Nama kolom:", list(data.columns))

# 2. Memeriksa kolom target
if "target_direction" not in data.columns:
    print("\nKolom 'target_direction' tidak ditemukan.")
    print("Periksa kembali dataset yang diunduh dari Kaggle.")
    raise SystemExit

# 3. Mengurutkan data berdasarkan tanggal jika kolom tanggal tersedia
kolom_tanggal = None
for kolom in data.columns:
    if kolom in ["date", "datetime", "timestamp", "time"]:
        kolom_tanggal = kolom
        break

if kolom_tanggal is not None:
    data[kolom_tanggal] = pd.to_datetime(data[kolom_tanggal], errors="coerce")
    data = data.sort_values(kolom_tanggal).reset_index(drop=True)

# 4. Memilih fitur yang digunakan
# Variasi nama kolom seperti lag_1, lag 1, atau lag1 akan dikenali.
fitur_dicari = {
    "open": ["open"],
    "high": ["high"],
    "low": ["low"],
    "close": ["close"],
    "returns": ["returns", "return"],
    "lag 1": ["lag1", "lag_1", "lag 1"],
    "lag 2": ["lag2", "lag_2", "lag 2"],
    "lag 3": ["lag3", "lag_3", "lag 3"]
}

def rapikan_nama(nama):
    return str(nama).lower().replace("_", "").replace(" ", "").replace("-", "")

fitur = []
for nama_baku, pilihan in fitur_dicari.items():
    for kolom in data.columns:
        if rapikan_nama(kolom) in [rapikan_nama(x) for x in pilihan]:
            fitur.append(kolom)
            break

if len(fitur) == 0:
    print("\nFitur yang diperlukan tidak ditemukan.")
    print("Pastikan dataset memiliki kolom open, high, low, close, returns, dan/atau lag.")
    raise SystemExit

# Jangan memasukkan kolom target atau informasi masa depan sebagai fitur.
# target_next_close sengaja tidak digunakan agar tidak terjadi data leakage.
print("\nFitur yang digunakan:", fitur)

data["target_direction"] = pd.to_numeric(data["target_direction"], errors="coerce")
data = data.dropna(subset=["target_direction"]).copy()
data = data[data["target_direction"].isin([0, 1])].copy()

for kolom in fitur:
    data[kolom] = pd.to_numeric(data[kolom], errors="coerce")

data = data.dropna(subset=fitur).reset_index(drop=True)

if len(data) < 10:
    print("Data yang bisa digunakan terlalu sedikit untuk pelatihan dan pengujian.")
    raise SystemExit

X = data[fitur]
y = data["target_direction"].astype(int)

# 5. Membagi data berdasarkan urutan waktu, tanpa mengacak data
batas = int(len(data) * 0.8)
X_latih = X.iloc[:batas]
X_uji = X.iloc[batas:]
y_latih = y.iloc[:batas]
y_uji = y.iloc[batas:]

if y_latih.nunique() < 2:
    print("Data latih hanya memiliki satu kelas target. Periksa dataset.")
    raise SystemExit

print("\n=== PEMBAGIAN DATA ===")
print("Data latih:", len(X_latih))
print("Data uji:", len(X_uji))

# 6. Melatih Random Forest
random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced"
)
random_forest.fit(X_latih, y_latih)
hasil_rf = random_forest.predict(X_uji)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



File 'gold.csv' tidak ditemukan.
Unduh dataset Kaggle, lalu ubah nama file menjadi gold.csv.
Traceback (most recent call last):
  File "/tmp/ipykernel_658/3951733170.py", line 26, in <cell line: 0>
    data = pd.read_csv(nama_file)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1880, in _make_engine
    self.handles = get_handle(
                   ~~~~~~~~~~^
        f,
        ^^
    ...<6 lines>...
        storage_options=self.options.get("storage_o

TypeError: object of type 'NoneType' has no len()